# Classical ML Model Selection and Training (Local)

In this notebook, we will focus exclusively on Phase 1 testing and selection for Classical Machine Learning:
- Validate heavy ML Baselines (Random Forest, SVM, Logistic Regression, KNN, Gaussian Naive Bayes).
- Compare models on Raw Pixels vs Handcrafted Features (Color Histograms, LBP, GLCM, HOG).
- Execute runs using the `train_ml.py`.

# Environment Setup

In [1]:
!git clone https://github.com/qu1r0ra/jute-disease-detection.git
%cd jute-disease-detection

%pip install uv
!uv pip install --system -e .
!uv sync

fatal: destination path 'jute-disease-detection' already exists and is not an empty directory.
/content/jute-disease-detection
Using Python 3.12.12 environment at: /usr
Resolved 132 packages in 896ms                                       
Prepared 1 package in 754ms                                              
Uninstalled 1 package in 1ms
Installed 1 package in 1msion==0.1.0 (from file:///content/j
 ~ jute-disease-detection==0.1.0 (from file:///content/jute-disease-detection)
Resolved 185 packages in 2ms
Audited 178 packages in 4ms


If you encounter `ModuleNotFoundError`, you can simply restart the session and rerun the cell below.

In [2]:
# ruff: noqa: T201
from jute_disease.utils.constants import DEFAULT_SEED
from jute_disease.utils.seed import seed_everything

seed_everything(DEFAULT_SEED)

INFO: Seed set to 42
INFO:lightning.fabric.utilities.seed:Seed set to 42


Mount your Google Drive to the Colab runtime.

In [3]:
from google.colab import drive

drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


If you haven't yet,

1. Download `data.zip` from <https://drive.google.com/drive/folders/1WoQ-Xzy0Prl9lInHW5JpGX4tpE9YDUua?usp=sharing> and upload it to your Google Colab account's Google Drive. You can simply upload it to the root of _My Drive_ for simplicity.
2. Update `DATA_ZIP_PATH` below to the path where you stored the file. If you uploaded it to the root of _My Drive_, you can set it to **"/content/drive/MyDrive/data.zip"**.

In [4]:
%cd jute-disease-detection

[Errno 2] No such file or directory: 'jute-disease-detection'
/content/jute-disease-detection


In [5]:
from pathlib import Path

# Update DATA_ZIP_PATH to where data.zip is stored relative to the Colab VM filesystem.
# For organization, we stored ours in!uv pip install --system -e .
# "/content/drive/MyDrive/Colab Notebooks/Jute Leaf Disease/data.zip"
DATA_ZIP_PATH = "/content/drive/MyDrive/data.zip"
DEST_PATH = Path("data/by_class")

if Path(DATA_ZIP_PATH).exists():
    DEST_PATH.mkdir(parents=True, exist_ok=True)
    print(f"Unzipping {DATA_ZIP_PATH} to {DEST_PATH}...")
    !unzip -q -n "$DATA_ZIP_PATH" -d "$DEST_PATH"
    print("Data unpacked.")
else:
    print(
        f"Zip file not found at {DATA_ZIP_PATH}. "
        "Please check the path or upload your data."
    )

Unzipping /content/drive/MyDrive/data.zip to data/by_class...
Data unpacked.


Let's cleanly construct the `train`, `val`, and `test` sub-folders inside `data/ml_split/` from your unzipped files.

In [6]:
!uv run python src/jute_disease/data/utils.py split

Seed set to 42
2026-03-08 03:06:27,456 - jute_disease.utils.seed - INFO - Random seed set to: 42
2026-03-08 03:06:27,457 - __main__ - INFO - Split directory /content/jute-disease-detection/data/ml_split already exists. Skipping split.


To persist our training artifacts beyond the Colab VM, we can _symlink_ the `artifacts` folder directly to our Google Drive.

In [8]:
GDRIVE_PATH = Path(DATA_ZIP_PATH).parent
GDRIVE_ARTIFACTS = GDRIVE_PATH / "artifacts"
GDRIVE_ARTIFACTS.mkdir(parents=True, exist_ok=True)

LOCAL_ARTIFACTS = Path("artifacts")

if not LOCAL_ARTIFACTS.exists() and not LOCAL_ARTIFACTS.is_symlink():
    LOCAL_ARTIFACTS.symlink_to(GDRIVE_ARTIFACTS)
    print(f"Symlinked {LOCAL_ARTIFACTS.absolute()} to {GDRIVE_ARTIFACTS}")
else:
    print(f"{LOCAL_ARTIFACTS} already exists or is linked.")

artifacts already exists or is linked.


Let us perform a quick sanity test to ensure all generated files show up inside your Google Drive folder containing your `data.zip`. If you see a generated `test.txt` file then you are all set to proceed.

In [9]:
test_file = LOCAL_ARTIFACTS / "test.txt"
test_file.write_text("ML model selection test.")

if (GDRIVE_ARTIFACTS / "test.txt").exists():
    print("Symlink worked.")
else:
    print("Symlink failed :<")

Symlink worked.


# Model Training Setup

We will now proceed to training the baseline models for this project. We will be using a variety of techniques in order to preprocess and supplement our `by_class` data. This preprocessing step consists of (1) data cleaning and (2) data augmentations. Thankfully, these two processes are already handled in the `src/jute_disease/data/transforms.py`, wherein the images are transformed to be set to 256x256 pixels. Additionally, it augments our image data using the **Albumentation** library in order to perform the following set of transformations:
- Cropping 
- Geometry (Rotation and Shear)
- Dropout (randomly drops rectangular regions from the image)
- Color and Lighting (Brightness and Contrast)
- Noise and Blur (Grain and Blur)

The `transforms.py` routine is called when training a model, hence we may jump straight into the training process. 



In [ ]:
# Better to add a sanity check or a preview of these transformations


To do this, we will be conducting experiments in classifying the images from the `by_class` folder using common image classification models. Specifically, the Classical ML models that we will be using are as follows:
- K-Nearest Neighbors (KNN)
- Logistic Regression (LR)
- Gaussian Naive Bayes (GNB)
- Random Forests (RF)
- Support Vector Machines (SVM)

The training process will consist of three key phases:
1. Feature Extraction
2. Classical Model Baselines
3. Results and Model Selection

## 1. Feature Extraction Pipeline
The feature extractions process consists of two types of extractions:
- **Raw**: flattened pixels
- **Crafted**: HOG, LBP, Color Moments

We rely on `jute_disease.models.ml.features` to extract and cache these datasets.


## 2. Classical Model Baselines

To assess the model baselines, we will be running our `train all_ml` script that handles the entire training process of the classical Ml models as follows:
Transform and Standardize Data -> Extract Features -> Train the size models -> Save Results to `WANDB`

In [ ]:
import os
# CHANGE THIS
os.environ["WANDB_API_KEY"] = "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"

!uv run python scripts/train_all_ml.py

2026-03-08 03:34:58,550 - __main__ - INFO - Starting ML Training Pipeline — 10 experiments.
2026-03-08 03:34:58,550 - __main__ - INFO - Training gnb with crafted features...
Seed set to 42
2026-03-08 03:35:08,968 - jute_disease.utils.seed - INFO - Random seed set to: 42
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: immanumali12 (immanumali12-de-la-salle-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: ⢿ Waiting for wandb.init()...
wandb: ⣻ Waiting for wandb.init()...
wandb: Tracking run with wandb version 0.25.0
wandb: Run data is saved locally in /content/jute-disease-detection/wandb/run-20260308_033509-hvobr5ip
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run gnb-crafted
wandb: ⭐️ View project at https://wandb.ai/grade-descent/jute-disease-detectio

## 3. Results and Model Selection
Using the metrics described from Weights & Biases, we can see that the best performing baseline models were SVM and RF in terms of accuracy. These two models will then proceed to model analaysis and tuning stage in the next notebook. 